# LeetCode 1528 — Shuffle String bằng Python

**Link đề:** https://leetcode.com/problems/shuffle-string/
**Độ khó:** Easy · **Chủ đề:** Array, String

```python
class Solution:
    def restoreString(self, s: str, indices: list[int]) -> str:
        ...
```

### Mục tiêu học
1. Hiểu đúng câu "ký tự ở vị trí `i` **di chuyển tới** vị trí `indices[i]`" — phân biệt *đi tới* và *lấy từ*.
2. Nắm chắc: chuỗi Python là **bất biến** (immutable), vì sao phải dùng **list ký tự** rồi `"".join(...)`.
3. Dùng thành thạo `[None] * n`, `enumerate`, `zip`, `sorted(..., key=...)`.
4. Đi qua 4 lời giải: dò tìm `O(n²)` → mảng kết quả `O(n)` → sắp xếp cặp `O(n log n)` → hoán vị tại chỗ theo chu trình.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, tự luyện biến thể (hoán vị ngược, kiểm tra hoán vị hợp lệ).

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — xáo chuỗi theo mảng chỉ số

Cho chuỗi `s` và mảng số nguyên `indices` **cùng độ dài** `n`. Chuỗi được xáo sao cho **ký tự ở vị trí `i` của `s` chuyển tới vị trí `indices[i]`** trong chuỗi kết quả. Trả về chuỗi sau khi xáo.

| Ví dụ | `s` | `indices` | Output |
|---|---|---|---|
| 1 | `"codeleet"` | `[4,5,6,7,0,2,1,3]` | `"leetcode"` |
| 2 | `"abc"` | `[0,1,2]` | `"abc"` |

**Constraints**
- `s.length == indices.length == n`, `1 <= n <= 100`
- `s` chỉ gồm chữ cái thường tiếng Anh.
- `0 <= indices[i] < n` và các giá trị của `indices` **đôi một khác nhau**.

Hai ràng buộc cuối nói rằng `indices` là một **hoán vị** của `0, 1, ..., n-1`: mỗi ô của kết quả nhận **đúng một** ký tự — không ô nào bị trống, không ô nào bị ghi hai lần.

Câu hỏi gợi mở:
- Với ví dụ 1, ký tự `'c'` (vị trí 0) đi đâu? Ký tự nào đến vị trí 0?
- Nếu đọc nhầm thành "kết quả ở vị trí `i` lấy `s[indices[i]]`" thì ví dụ 1 ra gì?

In [1]:
s = "codeleet"
indices = [4, 5, 6, 7, 0, 2, 1, 3]
print("n =", len(s), "| len(indices) =", len(indices))
for i in range(len(s)):
    print(f"s[{i}] = '{s[i]}'  ->  đi tới vị trí {indices[i]}")

n = 8 | len(indices) = 8
s[0] = 'c'  ->  đi tới vị trí 4
s[1] = 'o'  ->  đi tới vị trí 5
s[2] = 'd'  ->  đi tới vị trí 6
s[3] = 'e'  ->  đi tới vị trí 7
s[4] = 'l'  ->  đi tới vị trí 0
s[5] = 'e'  ->  đi tới vị trí 2
s[6] = 'e'  ->  đi tới vị trí 1
s[7] = 't'  ->  đi tới vị trí 3


Đọc bảng trên: `'c'` (vị trí 0) đi tới ô 4, `'l'` (vị trí 4) đi tới ô 0, ... Ghép theo ô 0..7: ô 0 = `'l'`, ô 1 = `'e'` (từ vị trí 6), ô 2 = `'e'` (vị trí 5), ô 3 = `'t'` (vị trí 7), ô 4 = `'c'`, ... → `"leetcode"`.

## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi là bất biến (immutable)

Ta **không thể** gán `t[2] = 'x'` cho một chuỗi. Python báo `TypeError`. Muốn "điền từng ô", ta dùng **list** (có thể sửa) rồi mới ghép lại.

In [2]:
t = "abc"
try:
    t[1] = "X"
except TypeError as e:
    print("Lỗi:", e)

chars = list(t)          # ['a', 'b', 'c'] — list thì sửa được
chars[1] = "X"
print(chars)

Lỗi: 'str' object does not support item assignment
['a', 'X', 'c']


`try/except` ở trên chỉ để **xem lỗi mà không dừng notebook**: chạy khối `try`, nếu phát sinh `TypeError` thì nhảy vào `except` và in thông báo.

### 2.2. Tạo list có sẵn `n` ô: `[None] * n`

`[None] * 4` tạo list 4 phần tử đều là `None` (giá trị "chưa có gì"). Đây là "khung" để ta điền ký tự vào đúng ô.

In [3]:
n = 4
res = [None] * n
print(res)
res[2] = "z"
res[0] = "a"
print(res)
print("Ô 1 còn trống?", res[1] is None)

[None, None, None, None]
['a', None, 'z', None]
Ô 1 còn trống? True


### 2.3. Ghép list ký tự thành chuỗi: `"".join(list)`

`sep.join(items)` nối các chuỗi trong `items`, chèn `sep` ở giữa. Với `sep = ""` ta nối liền.

Vì sao không cộng chuỗi `t = t + ch` trong vòng lặp? Mỗi phép `+` tạo chuỗi **mới** và chép lại toàn bộ, nên cộng dồn `n` lần có thể tốn `O(n²)`. `join` chỉ chép một lần → `O(n)`.

In [4]:
chars = ["l", "e", "e", "t"]
print("".join(chars))
print("-".join(chars))
print(", ".join(["a", "b", "c"]))

leet
l-e-e-t
a, b, c


### 2.4. `enumerate` và `zip`

- `enumerate(s)` cho từng cặp `(i, s[i])` — vừa có chỉ số vừa có giá trị.
- `zip(a, b)` ghép song song: `(a[0], b[0]), (a[1], b[1]), ...`. Rất hợp khi hai mảng cùng độ dài như `s` và `indices`.

In [5]:
s = "abc"
indices = [2, 0, 1]
for i, ch in enumerate(s):
    print("enumerate:", i, ch)
for ch, pos in zip(s, indices):
    print("zip:", ch, "->", pos)

enumerate: 0 a
enumerate: 1 b
enumerate: 2 c
zip: a -> 2
zip: b -> 0
zip: c -> 1


### 2.5. `sorted(..., key=...)`

`sorted(iterable, key=f)` trả về list mới, sắp theo giá trị `f(phần_tử)`. `lambda p: p[0]` là một hàm ngắn "lấy phần tử đầu của cặp `p`". Ta sẽ dùng nó ở cách 3.

In [6]:
pairs = [(2, "a"), (0, "b"), (1, "c")]
print(sorted(pairs, key=lambda p: p[0]))
print("Lấy cột ký tự:", [ch for _, ch in sorted(pairs)])

[(0, 'b'), (1, 'c'), (2, 'a')]
Lấy cột ký tự: ['b', 'c', 'a']


Dòng cuối dùng **list comprehension** `[biểu_thức for ... in ...]` — cách viết gọn của vòng `for` + `append`. Dấu `_` là tên biến "không dùng tới".

## 3. Cách trực giác — dò xem ký tự nào đến từng ô

Người mới thường nghĩ theo **ô kết quả**: "Ô `j` của kết quả là ký tự nào?" Trả lời: là ký tự `s[i]` mà `indices[i] == j`. Vậy với mỗi ô `j`, ta **dò** khắp `indices` để tìm `i`.

In [7]:
def restore_search(s: str, indices: list[int]) -> str:
    n = len(s)
    out = []
    for j in range(n):              # ô kết quả thứ j
        for i in range(n):          # tìm i có indices[i] == j
            if indices[i] == j:
                out.append(s[i])
                break
    return "".join(out)

print(restore_search("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))
print(restore_search("abc", [0, 1, 2]))

leetcode
abc


### 3.1. Chi phí của cách dò tìm

Với mỗi ô `j` ta có thể phải dò tới `n` phần tử → tệ nhất khoảng `1 + 2 + ... + n = n(n+1)/2` phép so sánh, tức `O(n²)`. Với `n = 100` vẫn chạy nhanh, nhưng ta đang làm việc **thừa**: thông tin "ký tự `i` đi đâu" đã có sẵn trong `indices[i]`, chẳng cần dò!

Đếm số lần so sánh để thấy rõ:

In [8]:
def count_compares(indices):
    n = len(indices); cnt = 0
    for j in range(n):
        for i in range(n):
            cnt += 1
            if indices[i] == j:
                break
    return cnt

for n in [4, 8, 16, 32]:
    rev = list(range(n - 1, -1, -1))   # trường hợp xấu: đảo ngược
    print(f"n={n:>2}: số so sánh = {count_compares(rev)}  (n(n+1)/2 = {n*(n+1)//2})")

n= 4: số so sánh = 10  (n(n+1)/2 = 10)
n= 8: số so sánh = 36  (n(n+1)/2 = 36)
n=16: số so sánh = 136  (n(n+1)/2 = 136)
n=32: số so sánh = 528  (n(n+1)/2 = 528)


## 4. Nút thắt và bước đột phá

**Nút thắt:** cách 3 hỏi "ô `j` lấy ai?" — câu hỏi đó cần **dò ngược**.

**Đột phá:** đổi góc nhìn sang **ký tự nguồn**: "ký tự `s[i]` đi đâu?" — câu trả lời có sẵn: `indices[i]`. Vậy chỉ cần:

```text
res[indices[i]] = s[i]     với mọi i
```

Đây là phép **scatter** (rải): mỗi phần tử tự đi về đúng chỗ. Vì `indices` là hoán vị, mỗi ô được ghi **đúng một lần**.

Kiểm tra bằng tay một bước:

In [9]:
s = "codeleet"; indices = [4, 5, 6, 7, 0, 2, 1, 3]
res = [None] * len(s)
i = 0
res[indices[i]] = s[i]
print(f"i=0: s[0]='{s[0]}' đặt vào ô {indices[0]} ->", res)
i = 4
res[indices[i]] = s[i]
print(f"i=4: s[4]='{s[4]}' đặt vào ô {indices[4]} ->", res)

i=0: s[0]='c' đặt vào ô 4 -> [None, None, None, None, 'c', None, None, None]
i=4: s[4]='l' đặt vào ô 0 -> ['l', None, None, None, 'c', None, None, None]


## 5. Lời giải chính — mảng kết quả (scatter)

### 5.1. Từng đoạn
1. `n = len(s)` và `res = [""] * n` — tạo khung `n` ô.
2. Duyệt `i` từ 0 đến `n-1`: `res[indices[i]] = s[i]`.
3. `return "".join(res)`.

### 5.2. Ghép lại

In [10]:
def restore_scatter(s: str, indices: list[int]) -> str:
    n = len(s)
    res = [""] * n                 # khung n ô
    for i in range(n):
        res[indices[i]] = s[i]     # ký tự i đi tới ô indices[i]
    return "".join(res)

print(restore_scatter("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))
print(restore_scatter("abc", [0, 1, 2]))
print(restore_scatter("aiohn", [3, 1, 4, 2, 0]))

leetcode
abc
nihao


Viết gọn hơn với `zip` — cùng ý tưởng, không cần chỉ số `i`:

In [11]:
def restore_scatter_zip(s: str, indices: list[int]) -> str:
    res = [""] * len(s)
    for ch, pos in zip(s, indices):
        res[pos] = ch
    return "".join(res)

print(restore_scatter_zip("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))

leetcode


## 6. TRACE từng bước

Ví dụ `s = "aiohn"`, `indices = [3,1,4,2,0]` (kết quả mong đợi `"nihao"`):

| i | `s[i]` | `indices[i]` | `res` sau bước |
|---|---|---|---|
| 0 | a | 3 | `_ _ _ a _` |
| 1 | i | 1 | `_ i _ a _` |
| 2 | o | 4 | `_ i _ a o` |
| 3 | h | 2 | `_ i h a o` |
| 4 | n | 0 | `n i h a o` |

Cột `res` cho thấy: mỗi bước điền **một ô mới**, không bao giờ ghi đè (vì `indices` không trùng).

In [12]:
def trace_restore(s, indices):
    res = ["_"] * len(s)
    print("Bắt đầu:", " ".join(res))
    for i, (ch, pos) in enumerate(zip(s, indices)):
        assert res[pos] == "_", "ô đã bị ghi trước đó!"
        res[pos] = ch
        print(f"i={i}: '{ch}' -> ô {pos} | res = {' '.join(res)}")
    ans = "".join(res)
    print("Kết quả:", ans)
    return ans

assert trace_restore("aiohn", [3, 1, 4, 2, 0]) == "nihao"
print()
assert trace_restore("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]) == "leetcode"

Bắt đầu: _ _ _ _ _
i=0: 'a' -> ô 3 | res = _ _ _ a _
i=1: 'i' -> ô 1 | res = _ i _ a _
i=2: 'o' -> ô 4 | res = _ i _ a o
i=3: 'h' -> ô 2 | res = _ i h a o
i=4: 'n' -> ô 0 | res = n i h a o
Kết quả: nihao

Bắt đầu: _ _ _ _ _ _ _ _
i=0: 'c' -> ô 4 | res = _ _ _ _ c _ _ _
i=1: 'o' -> ô 5 | res = _ _ _ _ c o _ _
i=2: 'd' -> ô 6 | res = _ _ _ _ c o d _
i=3: 'e' -> ô 7 | res = _ _ _ _ c o d e
i=4: 'l' -> ô 0 | res = l _ _ _ c o d e
i=5: 'e' -> ô 2 | res = l _ e _ c o d e
i=6: 'e' -> ô 1 | res = l e e _ c o d e
i=7: 't' -> ô 3 | res = l e e t c o d e
Kết quả: leetcode


`enumerate(zip(s, indices))` cho `(i, (ch, pos))` — vì thế phải viết `for i, (ch, pos) in ...` với ngoặc để **unpack lồng**.

## 7. Các lời giải khác

### 7.1. Cách 3 — Sắp xếp cặp `(indices[i], s[i])`

Ghép mỗi ký tự với "vị trí đích" của nó, rồi **sắp theo vị trí đích**. Sau khi sắp, ký tự đứng theo đúng thứ tự ô 0, 1, 2, ...

In [13]:
def restore_sort(s: str, indices: list[int]) -> str:
    pairs = sorted(zip(indices, s))       # sắp theo indices (phần tử đầu)
    return "".join(ch for _, ch in pairs)

print(sorted(zip([3, 1, 4, 2, 0], "aiohn")))
print(restore_sort("aiohn", [3, 1, 4, 2, 0]))
print(restore_sort("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))

[(0, 'n'), (1, 'i'), (2, 'h'), (3, 'a'), (4, 'o')]
nihao
leetcode


Cách này đẹp nhưng tốn `O(n log n)` do sắp xếp — chậm hơn scatter `O(n)` về lý thuyết. Ưu điểm: vẫn đúng ngay cả khi `indices` **không liên tục** (ví dụ `[10, 3, 7]`), chỉ cần thứ tự tương đối — một biến thể hay gặp.

### 7.2. Cách 4 — Hoán vị tại chỗ theo chu trình (cycle sort)

Câu hỏi phỏng vấn hay gặp: "Nếu không được tạo mảng `res` mới thì sao?" (Python vẫn phải đổi chuỗi sang list, nhưng ta có thể chỉ dùng **một** list và tráo đổi.)

Ý tưởng: tại vị trí `i`, khi `indices[i] != i`, ký tự đang ở `i` **chưa về đúng chỗ** → tráo nó với ô `indices[i]` (cả ký tự lẫn chỉ số). Mỗi lần tráo đưa **ít nhất một** ký tự về đúng chỗ vĩnh viễn → tổng số lần tráo ≤ `n - 1`.

In [14]:
def restore_cycle(s: str, indices: list[int]) -> str:
    chars = list(s)
    idx = indices[:]                 # copy để không làm hỏng input
    for i in range(len(chars)):
        while idx[i] != i:
            j = idx[i]
            chars[i], chars[j] = chars[j], chars[i]
            idx[i], idx[j] = idx[j], idx[i]
    return "".join(chars)

print(restore_cycle("aiohn", [3, 1, 4, 2, 0]))
print(restore_cycle("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))

nihao
leetcode


Cú pháp `a, b = b, a` là **tráo đổi đồng thời** của Python: vế phải được tính xong trước rồi mới gán. Và `indices[:]` là **slice toàn bộ** → tạo bản sao list.

Trace cách tráo đổi để thấy từng ký tự "về nhà":

In [15]:
def trace_cycle(s, indices):
    chars = list(s); idx = indices[:]
    for i in range(len(chars)):
        while idx[i] != i:
            j = idx[i]
            chars[i], chars[j] = chars[j], chars[i]
            idx[i], idx[j] = idx[j], idx[i]
            print(f"i={i}: tráo với ô {j} -> chars={''.join(chars)} idx={idx}")
    return "".join(chars)

assert trace_cycle("aiohn", [3, 1, 4, 2, 0]) == "nihao"

i=0: tráo với ô 3 -> chars=hioan idx=[2, 1, 4, 3, 0]
i=0: tráo với ô 2 -> chars=oihan idx=[4, 1, 2, 3, 0]
i=0: tráo với ô 4 -> chars=nihao idx=[0, 1, 2, 3, 4]


### 7.3. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Dò tìm | ô `j` lấy `s[i]` với `indices[i]==j` | `O(n²)` | `O(n)` | dễ nghĩ nhất, thừa việc |
| 2. Scatter | `res[indices[i]] = s[i]` | `O(n)` | `O(n)` | **nên dùng** |
| 3. Sắp xếp cặp | `sorted(zip(indices, s))` | `O(n log n)` | `O(n)` | đúng cả khi chỉ số không liên tục |
| 4. Chu trình | tráo đến khi `idx[i]==i` | `O(n)` | `O(n)` cho list ký tự (thuật toán dùng `O(1)` thêm nếu được sửa input) | mẹo phỏng vấn |

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def restoreString(self, s: str, indices: list[int]) -> str:
        res = [""] * len(s)
        for ch, pos in zip(s, indices):
            res[pos] = ch
        return "".join(res)

sol = Solution()
print(sol.restoreString("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))
print(sol.restoreString("abc", [0, 1, 2]))

leetcode
abc


Phiên bản sắp xếp (một dòng) cũng được chấp nhận:

```python
class Solution:
    def restoreString(self, s: str, indices: list[int]) -> str:
        return "".join(ch for _, ch in sorted(zip(indices, s)))
```

## 9. Vì sao thuật toán đúng?

**Đặc tả:** kết quả `t` thỏa `t[indices[i]] = s[i]` với mọi `i`.

**Invariant của vòng lặp scatter:** sau khi xử lý xong `i = 0..k-1`, các ô `indices[0..k-1]` của `res` đã chứa đúng ký tự `s[0..k-1]`, các ô còn lại chưa được ghi.

- Bước `k`: ô `indices[k]` **chưa từng bị ghi** (vì `indices` không trùng), nên ghi `s[k]` không phá kết quả cũ → invariant giữ.
- Kết thúc: đã ghi `n` ô khác nhau trong `n` ô → **mọi ô** đều có đúng ký tự yêu cầu. Không ô nào còn `""`, nên `join` cho đúng độ dài `n`.

Nếu `indices` có trùng, invariant gãy (một ô bị ghi đè, một ô bị trống) — đó là lý do đề cam kết "All values of indices are unique".

## 10. Phân tích Time / Space

Gọi `n = len(s)`.

- **Time:** vòng lặp `n` bước, mỗi bước gán `O(1)`; `join` `O(n)` → tổng `O(n)`.
- **Space:** list `res` `n` ô + chuỗi kết quả → `O(n)`. Không thể tránh vì phải trả về chuỗi mới độ dài `n`.

So với cách dò tìm `O(n²)`: với `n = 100` chênh lệch ~100 lần số phép so sánh — chưa đáng kể ở đây, nhưng thói quen "đổi góc nhìn để khỏi dò" rất quan trọng ở bài lớn.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.restoreString("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]), "leetcode")
check("đề 2", sol.restoreString("abc", [0, 1, 2]), "abc")
check("n = 1", sol.restoreString("z", [0]), "z")
check("đảo ngược", sol.restoreString("abcd", [3, 2, 1, 0]), "dcba")
check("xoay vòng", sol.restoreString("abcd", [1, 2, 3, 0]), "dabc")
check("ký tự lặp", sol.restoreString("aab", [2, 0, 1]), "aba")
check("nihao", sol.restoreString("aiohn", [3, 1, 4, 2, 0]), "nihao")

PASS đề 1: 'leetcode'
PASS đề 2: 'abc'
PASS n = 1: 'z'
PASS đảo ngược: 'dcba'
PASS xoay vòng: 'dabc'
PASS ký tự lặp: 'aba'
PASS nihao: 'nihao'


**Phản ví dụ cho cách đọc nhầm:** nếu viết `res[i] = s[indices[i]]` ("lấy từ") thay vì `res[indices[i]] = s[i]` ("đi tới"), ví dụ 2 và trường hợp đảo ngược vẫn đúng (vì hoán vị đối xứng), nhưng **xoay vòng** thì sai. Đây là lý do test phải có hoán vị **không đối xứng**.

In [18]:
def restore_wrong_gather(s, indices):
    return "".join(s[indices[i]] for i in range(len(s)))

for s, idx in [("abc", [0, 1, 2]), ("abcd", [3, 2, 1, 0]), ("abcd", [1, 2, 3, 0])]:
    print(s, idx, "| đúng:", restore_scatter(s, idx), "| đọc nhầm:", restore_wrong_gather(s, idx))

abc [0, 1, 2] | đúng: abc | đọc nhầm: abc
abcd [3, 2, 1, 0] | đúng: dcba | đọc nhầm: dcba
abcd [1, 2, 3, 0] | đúng: dabc | đọc nhầm: bcda


### 11.1. Đối chiếu 4 cách trên input ngẫu nhiên (có seed)

`random.Random(seed)` tạo bộ sinh ngẫu nhiên **lặp lại được**; `rng.shuffle(lst)` xáo list tại chỗ.

In [19]:
import random, string

rng = random.Random(1528)
for _ in range(500):
    n = rng.randint(1, 30)
    s = "".join(rng.choice(string.ascii_lowercase) for _ in range(n))
    idx = list(range(n)); rng.shuffle(idx)
    expected = restore_search(s, idx)
    assert restore_scatter(s, idx) == expected
    assert restore_scatter_zip(s, idx) == expected
    assert restore_sort(s, idx) == expected
    assert restore_cycle(s, idx) == expected
    assert Solution().restoreString(s, idx) == expected
print("500 test ngẫu nhiên: cả 4 cách cho cùng kết quả")

500 test ngẫu nhiên: cả 4 cách cho cùng kết quả


### 11.2. Tính chất: xáo rồi "xáo ngược" phải ra chuỗi ban đầu

Nếu `t = restore(s, indices)` thì `t[indices[i]] == s[i]`, tức là `s[i] = t[indices[i]]` → "lấy từ" `t` theo `indices` sẽ khôi phục `s`. Đây là **kiểm thử tính chất** (property test): không cần biết đáp án cụ thể.

In [20]:
rng = random.Random(7)
for _ in range(200):
    n = rng.randint(1, 20)
    s = "".join(rng.choice("abcxyz") for _ in range(n))
    idx = list(range(n)); rng.shuffle(idx)
    t = restore_scatter(s, idx)
    back = "".join(t[idx[i]] for i in range(n))
    assert back == s
print("Tính chất xáo ngược đúng trên 200 ca")

Tính chất xáo ngược đúng trên 200 ca


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Gán vào chuỗi
```python
res = " " * n
res[indices[i]] = s[i]      # TypeError: 'str' object does not support item assignment
```
**Sửa:** dùng list `[""] * n` rồi `join`.

### Lỗi 2 — Nhầm chiều "đi tới" với "lấy từ"
```python
res[i] = s[indices[i]]      # sai chiều
```
Đã thấy ở mục 11: sai với hoán vị xoay vòng. **Sửa:** `res[indices[i]] = s[i]`.

### Lỗi 3 — Cộng chuỗi theo thứ tự `i`
```python
t = ""
for i in range(n):
    t += s[indices[i]]       # vừa sai chiều, vừa cộng chuỗi
```
Cộng chuỗi luôn nối vào **cuối**, không thể "đặt vào ô k" → buộc phải dùng list.

### Lỗi 4 — Cycle sort nhưng dùng `if` thay vì `while`
Chỉ tráo một lần tại `i` có thể để lại ký tự khác chưa đúng chỗ ở `i`. Xem demo:

In [21]:
def restore_cycle_wrong(s, indices):
    chars = list(s); idx = indices[:]
    for i in range(len(chars)):
        if idx[i] != i:                  # SAI: chỉ tráo một lần
            j = idx[i]
            chars[i], chars[j] = chars[j], chars[i]
            idx[i], idx[j] = idx[j], idx[i]
    return "".join(chars)

s, idx = "abcd", [1, 2, 3, 0]
print("if   :", restore_cycle_wrong(s, idx))
print("while:", restore_cycle(s, idx), "(đúng)")

if   : badc
while: dabc (đúng)


Với `[1,2,3,0]` đây là **một chu trình dài 4**; tráo một lần tại `i=0` chỉ đưa `'a'` về ô 1, còn ô 0 nhận `'b'` vẫn chưa đúng. Phải `while` cho đến khi ô `i` ổn định.

### Lỗi 5 — Làm hỏng input khi cycle sort
Tráo trực tiếp trên `indices` (không copy) làm thay đổi mảng của người gọi. Trên LeetCode thường vô hại, nhưng trong code thật đây là **side effect** khó debug → dùng `indices[:]`.

## 13. Mẹo nhận dạng pattern

- Thấy câu "phần tử ở vị trí `i` **chuyển tới** vị trí `p[i]`" → **scatter**: `res[p[i]] = a[i]`.
- Thấy "vị trí `i` của kết quả **lấy** phần tử `p[i]`" → **gather**: `res[i] = a[p[i]]`.
- Scatter và gather là **ngược nhau**: scatter theo `p` = gather theo hoán vị nghịch đảo `p⁻¹`.
- Đề bảo "mọi giá trị khác nhau và nằm trong `[0, n)`" → đó là **hoán vị** → nghĩ tới chu trình, đánh dấu, đặt-đúng-chỗ (cyclic sort).
- Cần sửa "từng ô" của chuỗi → **list ký tự + join**.

## 14. Biến thể

### 14.1. Hoán vị nghịch đảo
Cho `p` là hoán vị, tìm `q` sao cho `q[p[i]] = i`. Đây chính là scatter với giá trị là chỉ số!

In [22]:
def inverse_permutation(p):
    q = [0] * len(p)
    for i, pos in enumerate(p):
        q[pos] = i
    return q

p = [4, 5, 6, 7, 0, 2, 1, 3]
q = inverse_permutation(p)
print("p =", p); print("q =", q)
s = "codeleet"
print("scatter theo p :", restore_scatter(s, p))
print("gather theo q  :", "".join(s[q[j]] for j in range(len(s))))

p = [4, 5, 6, 7, 0, 2, 1, 3]
q = [4, 6, 5, 7, 0, 1, 2, 3]
scatter theo p : leetcode
gather theo q  : leetcode


Hai dòng cuối bằng nhau: **scatter theo `p` ≡ gather theo `p⁻¹`**. Cách 1 (dò tìm) thực chất là đang tính `p⁻¹[j]` bằng tìm kiếm tuyến tính.

### 14.2. Kiểm tra `indices` có phải hoán vị hợp lệ
Nếu đề không cam kết, ta nên kiểm tra: độ dài đúng, mọi giá trị trong `[0, n)`, không trùng.

In [23]:
def is_permutation(idx):
    n = len(idx)
    seen = [False] * n
    for v in idx:
        if v < 0 or v >= n or seen[v]:
            return False
        seen[v] = True
    return True

for idx in [[2, 0, 1], [0, 0, 1], [0, 3, 1], [1, 0]]:
    print(idx, "->", is_permutation(idx))

[2, 0, 1] -> True
[0, 0, 1] -> False
[0, 3, 1] -> False
[1, 0] -> True


### 14.3. Đếm số chu trình của hoán vị
Số lần tráo tối thiểu để đưa mọi phần tử về chỗ = `n - (số chu trình)`. Ví dụ `[1,2,3,0]` có 1 chu trình → cần 3 lần tráo.

In [24]:
def count_cycles(p):
    seen = [False] * len(p); cycles = 0
    for i in range(len(p)):
        if not seen[i]:
            cycles += 1
            j = i
            while not seen[j]:
                seen[j] = True
                j = p[j]
    return cycles

for p in [[0, 1, 2], [1, 2, 3, 0], [4, 5, 6, 7, 0, 2, 1, 3]]:
    c = count_cycles(p)
    print(p, "| chu trình:", c, "| số tráo tối thiểu:", len(p) - c)

[0, 1, 2] | chu trình: 3 | số tráo tối thiểu: 0
[1, 2, 3, 0] | chu trình: 1 | số tráo tối thiểu: 3
[4, 5, 6, 7, 0, 2, 1, 3] | chu trình: 3 | số tráo tối thiểu: 5


## 15. Bài tập (tự làm — các cell TODO không được gọi khi chạy notebook)

### Bài 1 — Tính tay
Cho `s = "art"`, `indices = [1, 0, 2]`. Kết quả là gì? Viết bảng trace 3 dòng.

### Bài 2 — Đoán output
```python
res = [""] * 3
for ch, pos in zip("xyz", [2, 0, 1]):
    res[pos] = ch
print(res, "".join(res))
```

### Bài 3 — TODO: tự viết lời giải scatter

In [25]:
def exercise_restore(s: str, indices: list[int]) -> str:
    # TODO 1: tạo list res có len(s) ô
    # TODO 2: với mỗi (ch, pos), đặt ch vào res[pos]
    # TODO 3: ghép res thành chuỗi và trả về
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_restore("codeleet", [4,5,6,7,0,2,1,3]) == "leetcode"
# assert exercise_restore("abc", [0,1,2]) == "abc"

### Bài 4 — TODO: biến thể "lấy từ" (gather)
Cho `s` và `order`, kết quả ở vị trí `i` là `s[order[i]]`. Ví dụ `s="abc"`, `order=[2,0,1]` → `"cab"`.

In [26]:
def exercise_gather(s: str, order: list[int]) -> str:
    # TODO: res[i] = s[order[i]]
    pass

# assert exercise_gather("abc", [2, 0, 1]) == "cab"

### Bài 5 — TODO: chỉ số không liên tục
Cho `s` và `keys` (các số nguyên **khác nhau** bất kỳ, có thể rất lớn hoặc âm). Ký tự có `key` nhỏ hơn đứng trước. Ví dụ `s="abc"`, `keys=[50, -3, 7]` → `"bca"`.

In [27]:
def exercise_restore_keys(s: str, keys: list[int]) -> str:
    # TODO: không dùng được res[keys[i]] trực tiếp — vì sao?
    pass

# assert exercise_restore_keys("abc", [50, -3, 7]) == "bca"

### Bài 6 — Mini challenge: áp dụng hoán vị `k` lần
Áp dụng phép xáo `restoreString` với cùng `indices` lặp lại `k` lần (`k` có thể tới `10^9`). Viết hàm trả kết quả nhanh (gợi ý: chu trình).

In [28]:
def exercise_shuffle_k_times(s: str, indices: list[int], k: int) -> str:
    # TODO: mỗi ký tự đi vòng quanh chu trình của nó; đi k bước = đi k % (độ dài chu trình) bước
    pass

# assert exercise_shuffle_k_times("abcd", [1, 2, 3, 0], 1) == "dabc"
# assert exercise_shuffle_k_times("abcd", [1, 2, 3, 0], 4) == "abcd"
# assert exercise_shuffle_k_times("abcd", [1, 2, 3, 0], 10**9 + 1) == "dabc"

## 16. Gợi ý 3 mức

**Bài 1–2:** (1) Mỗi ký tự đi tới ô `pos`. (2) Điền từng ô theo bảng. (3) `"art"` → ô 1=`a`, ô 0=`r`, ô 2=`t`.

**Bài 3:** (1) Đổi góc nhìn: ký tự đi đâu? (2) `[""] * n`, `zip`, `join`. (3) `res[pos] = ch`.

**Bài 4:** (1) Đây là chiều ngược của bài gốc. (2) Duyệt `i` theo kết quả. (3) `"".join(s[order[i]] for i in range(len(s)))`.

**Bài 5:** (1) Chỉ số có thể âm/lớn → không làm ô list được. (2) Cái gì giữ được "thứ tự tương đối"? (3) `sorted(zip(keys, s))`.

**Bài 6:** (1) Sau một lần xáo, ký tự ở `i` tới `indices[i]`; sau 2 lần tới `indices[indices[i]]`. (2) Tách các chu trình như mục 14.3, lưu danh sách vị trí của mỗi chu trình. (3) Trong chu trình `cyc` độ dài `L`, ký tự ở `cyc[t]` sau `k` lần tới `cyc[(t + k) % L]`.

## 17. Đáp án tham khảo

**Bài 1:** `"rat"` — `a→ô1`, `r→ô0`, `t→ô2`.

**Bài 2:** in `['y', 'z', 'x'] yzx` — `x→ô2`, `y→ô0`, `z→ô1`.

In [29]:
def answer_restore(s, indices):
    res = [""] * len(s)
    for ch, pos in zip(s, indices):
        res[pos] = ch
    return "".join(res)

def answer_gather(s, order):
    return "".join(s[order[i]] for i in range(len(s)))

def answer_restore_keys(s, keys):
    return "".join(ch for _, ch in sorted(zip(keys, s)))

def answer_shuffle_k_times(s, indices, k):
    n = len(s); res = [""] * n; seen = [False] * n
    for start in range(n):
        if seen[start]:
            continue
        cyc = []; j = start
        while not seen[j]:
            seen[j] = True; cyc.append(j); j = indices[j]
        L = len(cyc)
        for t, pos in enumerate(cyc):
            res[cyc[(t + k) % L]] = s[pos]
    return "".join(res)

assert answer_restore("codeleet", [4,5,6,7,0,2,1,3]) == "leetcode"
assert answer_restore("abc", [0,1,2]) == "abc"
assert answer_restore("art", [1,0,2]) == "rat"
assert answer_gather("abc", [2, 0, 1]) == "cab"
assert answer_restore_keys("abc", [50, -3, 7]) == "bca"
assert answer_shuffle_k_times("abcd", [1, 2, 3, 0], 1) == "dabc"
assert answer_shuffle_k_times("abcd", [1, 2, 3, 0], 4) == "abcd"
assert answer_shuffle_k_times("abcd", [1, 2, 3, 0], 10**9 + 1) == "dabc"

# đối chiếu bài 6 với mô phỏng trực tiếp k lần (k nhỏ)
rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 10); s = "".join(rng.choice("abcde") for _ in range(n))
    idx = list(range(n)); rng.shuffle(idx); k = rng.randint(0, 12)
    t = s
    for _ in range(k):
        t = answer_restore(t, idx)
    assert answer_shuffle_k_times(s, idx, k) == t
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def restoreString(self, s: str, indices: list[int]) -> str:
        res = [""] * len(s)              # khung n ô (list, vì str bất biến)
        for ch, pos in zip(s, indices):
            res[pos] = ch                # "đi tới": scatter
        return "".join(res)              # ghép O(n)
```

**Câu thần chú:** *"Ký tự i đi tới ô indices[i]" → `res[indices[i]] = s[i]`.*

**Checklist trước khi submit**
- [ ] Đúng chiều: `res[indices[i]] = s[i]`, không phải `res[i] = s[indices[i]]`.
- [ ] Dùng list rồi `join`, không gán vào chuỗi.
- [ ] Test với hoán vị **không đối xứng** (ví dụ xoay vòng `[1,2,3,0]`).
- [ ] Trường hợp `n = 1`.
- [ ] Đúng tên method `restoreString` và chữ ký.